# ShiroRVC — vocoder pretraining on TPU (Kaggle): prepare + train

Preprocesses and extracts a raw audio dataset, then pretrains a ShiroRVC vocoder, all on a Kaggle TPU.
- **`tools/TPU/prepare_dataset_tpu.py`** preprocesses and extracts the dataset.
- **`tools/TPU/train_tpu.py`** trains the model.

If the dataset is already preprocessed and extracted, use `kaggle_tpu_train.ipynb` instead.

**Session settings:** accelerator **TPU v5e-8**, *Internet* on.

## Vocoders
| `VOCODER` | Sample rates | Discriminator | Spectral loss |
|---|---|---|---|
| `refinegan2` | 32 kHz | v4 + UnivHD + SAN | multi-scale mel |
| `hifi` | 32 / 40 / 48 kHz | v2 (8 periods + MSD) | L1 mel |
| `hifi++` | 32 kHz | v4 + UnivHD + SAN | multi-scale mel |

`hifi++` is disabled in the app's UI (`rvc/configs/vocoders.json`), but these scripts train it anyway. The architecture, sample rate and losses come from the experiment's `config.json`.

## Inputs
- **`RAW_DATASET_DIR`:** a Dataset with the audio files. For a multi-speaker set, use subfolders named `0_name`, `1_name`, …
- *(Optional)* **`PRETRAIN_G` / `PRETRAIN_D`:** a starting G/D pair. Without one, training starts from scratch.
- *(Optional)* **`RESUME_DIR`:** a previous session's `G_*.pth` / `D_*.pth`.

`tools/TPU` must exist in the repository the notebook clones. Push it, or upload the repository as a Dataset and set `REPO_DATASET`.

## How the dataset preparation uses the TPU
- **Preprocessing:** slicing, VAD and loudness are CPU work. They run on every host core, with one BLAS/OpenMP thread per worker.
- **Extraction on the TPU:** RMVPE and the embedder run there in fixed-size batches of clips with the *exact same length*. Padding would change the features, and fixed shapes mean XLA compiles once per length.
- **Rare lengths on the CPU:** lengths with fewer than `EXTRACT_MIN_GROUP` clips go to a CPU pool, so they cost no compilation.

## Disk
The sliced audio and the features are written under `WORK_DIR`. `/kaggle/working` is persisted but small. For a large dataset, set `WORK_DIR = "/kaggle/tmp"`: it is bigger but discarded when the session ends. Either way, the output package is written to `/kaggle/working`.

## Batch size and learning rate
The global batch is `BATCH_SIZE × 8 chips` (16 × 8 = 128 by default).
The config's learning rates are scaled by the **square root** of the ratio between the global batch and `REFERENCE_BATCH`.
The default of 16 is deliberately conservative. The shipped configs were tuned around batch 8, where the ratio would be 128/8 (×4) and a 2e-4 config would reach 8e-4, which is high for Adam on a GAN.
With 16, the ratio is 128/16 and the factor is ×2.83, so 2e-4 becomes about 5.7e-4. Set `REFERENCE_BATCH = 8` for the full √ rule.
Linear scaling tends to destabilise the G/D balance under Adam.
A linear warmup (default: the smaller of 1000 steps and 5% of the run) keeps the higher learning rate from breaking the first steps.
- **Out of memory at compile time:** use `BATCH_SIZE = 12` or `8`. The learning rate follows automatically.
- **Memory to spare:** `BATCH_SIZE = 24` or `32` uses the chips more.
- **HiFi-GAN at 40 or 48 kHz:** each item carries a longer training segment, so start lower there.
- **`loss_disc` collapses or `grad_norm` spikes:** set `LR_SCALE = 0.5`.

## Not ported to the TPU trainer
Holdout/overtrain detection, audio previews, freeze stages, and optimizers other than AdamW.

In [ ]:
# ===== Settings =====
REPO_URL = "https://github.com/ShiromiyaG/ShiroRVC"
REPO_BRANCH = "main"
REPO_DATASET = ""  # optional: the repository uploaded as a Dataset (used instead of git clone)
WORK_DIR = "/kaggle/working"  # "/kaggle/tmp" for datasets too large for /kaggle/working

MODEL_NAME = "pretrain"
RAW_DATASET_DIR = "/kaggle/input/my-audio"
VOCODER = "refinegan2"   # "refinegan2", "hifi" or "hifi++"
SAMPLE_RATE = 32000      # hifi also takes 40000 / 48000

PRETRAIN_G = ""  # optional: starting G
PRETRAIN_D = ""  # optional: starting D
RESUME_DIR = ""  # optional: folder with G_*.pth / D_*.pth from a previous session

# Dataset preparation (same meaning as core.py preprocess / extract)
CUT = "New Automatic"   # "Skip", "Simple", "Automatic" or "New Automatic"
CHUNK_LEN = 3.0
OVERLAP_LEN = 0.36
NORMALIZATION = "pre_peak_rvc"
NOISE_REDUCTION = False
EMBEDDER = "contentvec"  # "contentvec" or "spin_v2"
INCLUDE_MUTES = 5
FEATURE_PRECISION = "fp32"  # "fp16" halves the feature cache
EXTRACT_BATCH_SIZE = 32  # clips per chip per extraction step
EXTRACT_MIN_GROUP = 64   # smaller same-length groups are extracted on the CPU

# Training
EPOCHS = 100
BATCH_SIZE = 16          # per chip; global = 16 x 8 = 128
REFERENCE_BATCH = 16     # the LR factor is sqrt(global batch / this); 8 is the configs' own batch
LR_SCALING = "sqrt"      # "sqrt", "linear" or "none"
WARMUP_STEPS = -1        # -1 = min(1000, 5% of the run)
LR_SCALE = 1.0
SAVE_EVERY = 10          # epochs between checkpoints
SAVE_ONLY_LATEST = True
PRECISION = "bf16"       # "bf16" or "fp32"
MAX_FRAMES = 0           # 0 = sized from the dataset
NUM_WORKERS = 4          # DataLoader workers per chip; each one costs host RAM
LOG_EVERY = 50
SHOW_XLA_METRICS = False  # print recompilation metrics after the first epoch
SPMD = True              # one process for all 8 chips: one compile, far less host RAM
SPLIT_STEP = True        # four smaller graphs per step: much faster compile, slightly slower steps

## Code

In [ ]:
import os
import re
import shutil
import subprocess
import sys

REPO = f"{WORK_DIR}/ShiroRVC"
OUTPUT_DIR = "/kaggle/working"

if REPO_DATASET:
    shutil.copytree(REPO_DATASET, REPO, dirs_exist_ok=True)
elif not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO], check=True)
assert os.path.isfile(f"{REPO}/tools/TPU/train_tpu.py"), (
    "tools/TPU is not in this repository: push the folder or use REPO_DATASET."
)

## Environment
- **`torch_xla`:** installed only if the image lacks it.
- **`torchaudio`:** matched to the installed `torch`.
- **Everything else:** the rest of `requirements.txt` (preprocessing and extraction need most of it), minus the torch family and the UI-only packages.

In [ ]:
import importlib.util


def pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


if importlib.util.find_spec("torch_xla") is None:
    pip(
        "torch==2.8.0", "torch_xla[tpu]==2.8.0",
        "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
        "-f", "https://storage.googleapis.com/libtpu-wheels/index.html",
    )

import torch

torch_version = torch.__version__.split("+")[0]
try:
    import torchaudio
    torchaudio_ok = torchaudio.__version__.split(".")[:2] == torch_version.split(".")[:2]
except Exception:
    torchaudio_ok = False
if not torchaudio_ok:
    pip(f"torchaudio=={torch_version}", "--no-deps", "--index-url", "https://download.pytorch.org/whl/cpu")

SKIP = re.compile(r"^(torch|torchaudio|triton|triton-windows|gradio|edge-tts)\b", re.IGNORECASE)
with open(f"{REPO}/requirements.txt", encoding="utf-8") as handle:
    requirements = [
        line.split("#")[0].strip()
        for line in handle
        if line.split("#")[0].strip() and not SKIP.match(line.strip())
    ]
pip("setuptools", *requirements)

import torch_xla
print("torch", torch.__version__, "| torch_xla", torch_xla.__version__)

## Dataset

In [ ]:
import shlex

PREP_ARGS = shlex.join(str(a) for a in [
    "--model-name", MODEL_NAME,
    "--dataset", RAW_DATASET_DIR,
    "--sample-rate", SAMPLE_RATE,
    "--vocoder", VOCODER,
    "--cut", CUT,
    "--chunk-len", CHUNK_LEN,
    "--overlap-len", OVERLAP_LEN,
    "--normalization", NORMALIZATION,
    "--embedder", EMBEDDER,
    "--include-mutes", INCLUDE_MUTES,
    "--feature-precision", FEATURE_PRECISION,
    "--batch-size", EXTRACT_BATCH_SIZE,
    "--min-group", EXTRACT_MIN_GROUP,
    *(["--noise-reduction"] if NOISE_REDUCTION else []),
])
print(PREP_ARGS)

In [ ]:
!cd {REPO} && PJRT_DEVICE=TPU python tools/TPU/prepare_dataset_tpu.py {PREP_ARGS}

In [ ]:
MODEL_DIR = f"{REPO}/logs/{MODEL_NAME}"

In [ ]:
with open(f"{MODEL_DIR}/filelist.txt", encoding="utf-8") as handle:
    rows = [line.strip() for line in handle if line.strip()]
missing = [r.split("|")[0] for r in (rows[0], rows[-1]) if not os.path.isfile(f"{REPO}/{r.split('|')[0]}")]
assert not missing, f"Filelist entries not found: {missing}"

if RESUME_DIR:
    pairs = sorted(
        int(m.group(1))
        for m in map(re.compile(r"^G_(\d+)\.pth$").match, os.listdir(RESUME_DIR))
        if m and os.path.isfile(f"{RESUME_DIR}/D_{m.group(1)}.pth")
    )
    assert pairs, f"No G_/D_ pair in {RESUME_DIR}"
    for prefix in ("G", "D"):
        shutil.copy2(f"{RESUME_DIR}/{prefix}_{pairs[-1]}.pth", MODEL_DIR)
    print(f"Resuming from step {pairs[-1]}")

import json

sys.path.insert(0, REPO)
from rvc.configs.vocoders import get_vocoder_sample_rates, normalize_vocoder

with open(f"{MODEL_DIR}/config.json", encoding="utf-8") as handle:
    config = json.load(handle)
with open(f"{MODEL_DIR}/model_info.json", encoding="utf-8") as handle:
    recorded_vocoder = json.load(handle).get("vocoder_architecture", "")
assert VOCODER or recorded_vocoder, "model_info.json records no vocoder: set VOCODER."
VOCODER = normalize_vocoder(VOCODER or recorded_vocoder)
# config.json was copied from the recorded vocoder's folder, so another id would build from the wrong config.
assert not recorded_vocoder or normalize_vocoder(recorded_vocoder) == VOCODER, (
    f"The data was prepared for {recorded_vocoder!r}, not {VOCODER!r}: re-run the extraction for {VOCODER!r}."
)
rate = int(config["data"]["sample_rate"])
print("Model:", MODEL_NAME, "| vocoder:", VOCODER, "| sample rate:", rate, "| clips:", len(rows))
assert rate in get_vocoder_sample_rates(VOCODER), f"{VOCODER} has no configuration for {rate} Hz."

## Training
- **First epoch:** slow, because XLA compiles the step. Every later step reuses the same graph.
- **Later epochs still slow:** turn on `SHOW_XLA_METRICS` and check `CompileTime`, which should stop growing.
- **Scalars:** written to `logs/<model>/eval` (TensorBoard).

In [ ]:
args = [
    "--model-name", MODEL_NAME,
    "--vocoder", VOCODER,
    "--epochs", EPOCHS,
    "--batch-size", BATCH_SIZE,
    "--reference-batch", REFERENCE_BATCH,
    "--lr-scaling", LR_SCALING,
    "--warmup-steps", WARMUP_STEPS,
    "--lr-scale", LR_SCALE,
    "--save-every", SAVE_EVERY,
    "--precision", PRECISION,
    "--max-frames", MAX_FRAMES,
    "--num-workers", NUM_WORKERS,
    "--log-every", LOG_EVERY,
    "--no-weight-export",
]
if PRETRAIN_G:
    args += ["--pretrain-g", PRETRAIN_G]
if PRETRAIN_D:
    args += ["--pretrain-d", PRETRAIN_D]
if SAVE_ONLY_LATEST:
    args.append("--save-only-latest")
if SHOW_XLA_METRICS:
    args.append("--metrics")
args.append("--spmd" if SPMD else "--no-spmd")
if SPLIT_STEP:
    args.append("--split-step")
TRAIN_ARGS = shlex.join(str(a) for a in args)
print(TRAIN_ARGS)

In [ ]:
# Frees the TPU after a crashed or interrupted run: kills every Python process
# except Jupyter and this kernel. The last line should print 0.
import signal
import time

keep = {os.getpid(), os.getppid()}
for pid in map(int, filter(str.isdigit, os.listdir("/proc"))):
    try:
        cmd = open(f"/proc/{pid}/cmdline", "rb").read().replace(b"\0", b" ").decode(errors="ignore")
    except OSError:
        continue
    if pid in keep or "python" not in cmd or any(k in cmd for k in ("jupyter", "ipykernel", "notebook", "kaggle")):
        continue
    try:
        os.kill(pid, signal.SIGKILL)
        print("killed", pid, cmd[:80])
    except OSError:
        pass
time.sleep(3)
print("TPU handles still open:", os.popen("ls -l /proc/*/fd 2>/dev/null | grep -c vfio").read().strip())

In [ ]:
!cd {REPO} && PJRT_DEVICE=TPU python tools/TPU/train_tpu.py {TRAIN_ARGS}

## CUDA compatibility
`convert_to_cuda.py` checks that the latest `G_`/`D_` load strictly into the CUDA trainer's models. It then switches the AdamW state to its GPU flags, keeping the originals as `*.tpu.pth`.

No inference `.pth` is written.

## Pretrain
`make_pretrain.py` turns that pair into `<model>_G.pth` / `<model>_D.pth`, ready to pass as `pretrainG` / `pretrainD` on a GPU. Each file holds fp32 weights plus the metadata the pretrain checks read, with no optimizer state or counters, so it is much smaller than the checkpoint.
On your PC, `--install` puts them where the app looks for the vocoder's default pretrain.

In [ ]:
CONVERT_ARGS = f"--model-name {shlex.quote(MODEL_NAME)} --vocoder {shlex.quote(VOCODER)}"
!cd {REPO} && python tools/TPU/convert_to_cuda.py {CONVERT_ARGS}

In [ ]:
!cd {REPO} && python tools/TPU/make_pretrain.py {CONVERT_ARGS} --overwrite

## Output package

In [ ]:
import glob

OUT = f"{OUTPUT_DIR}/{MODEL_NAME}_tpu"
os.makedirs(OUT, exist_ok=True)
checkpoint_steps = sorted(
    int(m.group(1))
    for m in (re.match(r"^G_(\d+)\.pth$", os.path.basename(p)) for p in glob.glob(f"{MODEL_DIR}/G_*.pth"))
    if m
)
keep = []
if checkpoint_steps:
    keep += [f"{MODEL_DIR}/G_{checkpoint_steps[-1]}.pth", f"{MODEL_DIR}/D_{checkpoint_steps[-1]}.pth"]
keep += [f"{MODEL_DIR}/{MODEL_NAME}_G.pth", f"{MODEL_DIR}/{MODEL_NAME}_D.pth"]
keep += [f"{MODEL_DIR}/{name}" for name in ("config.json", "model_info.json")]
for path in keep:
    if os.path.isfile(path):
        shutil.copy2(path, OUT)
if os.path.isdir(f"{MODEL_DIR}/eval"):
    shutil.copytree(f"{MODEL_DIR}/eval", f"{OUT}/eval", dirs_exist_ok=True)
shutil.make_archive(OUT, "zip", OUT)
print(sorted(os.listdir(OUT)))
print("Archive:", OUT + ".zip")